# Econ 3916 PS3 — The Number Nobody Can Reproduce

**ECON 3916 · Applied Data Analytics** · Topics 9-11 · 100 points · your role: Analyst, Business Intelligence

This is the **starter**: the structure of the set in rubric order. Fill the empty cells; keep every heading so the grader finds each step. The brief on Canvas has the full text of every step, the numbers you must hit and the rubric. Every claim goes in a text cell: the reader is analyst's audience, not a grader with a kernel.

**AI is not allowed in Phases 1–3.** Generative AI is **prohibited** in Phases 1 through 3. Write the SQL yourself; that is the skill being assessed and it is the one interviewers test. AI is authorised and required in Phase 4.

## Setup

In [18]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)     # one generator, so every draw reruns the same way
pd.set_option('display.precision', 3)

## Phase 1: Reproduce Both Numbers (25 pts)

### Step 1.1: Build the warehouse (8 pts)

In [19]:
!pip install -q duckdb                      # the brief's setup step 2 (quick if Colab already has it)
import duckdb
import statsmodels.formula.api as smf       # for Phase 3

con = duckdb.connect()                       # your warehouse: register customers and orders here

# Parameters the brief fixes (Step 1.1). Keep them here so the grader can rerun you.
N_CUSTOMERS = 3_000          # at least 3,000 customers ...
MIN_WEST = 2_000             # ... and at least 2,000 of them in the West
NB_N, NB_P = 0.8, 0.12       # n_orders = 1 + NegativeBinomial(n=0.8, p=0.12): mean near 7, long right tail
REV_PER_ORDER = 60           # revenue = 60 * n_orders + Normal(400, 150), floored at 50
REV_MEAN, REV_SD, REV_FLOOR = 400, 150, 50

In [20]:
# Customers order count first
n_orders = 1 + rng.negative_binomial(NB_N, NB_P, size=N_CUSTOMERS)

n_other = (N_CUSTOMERS - MIN_WEST) // 2
region_labels = np.repeat(["West", "East", "Central"], [MIN_WEST, n_other, N_CUSTOMERS - MIN_WEST - n_other])
region = rng.choice(region_labels, size=N_CUSTOMERS, replace=False)

revenue = np.maximum(REV_PER_ORDER * n_orders + rng.normal(REV_MEAN, REV_SD, N_CUSTOMERS), REV_FLOOR).round(2)
signup_date = rng.choice(pd.date_range("2022-01-01", "2024-12-31"), size=N_CUSTOMERS)

customers_df = pd.DataFrame({
    "customer_id": np.arange(1, N_CUSTOMERS + 1),
    "region": region,
    "revenue": revenue,
    "signup_date": signup_date
})
orders_df = pd.DataFrame({"customer_id": np.repeat(customers_df["customer_id"].to_numpy(), n_orders)})
orders_df.insert(0, "order_id", np.arange(1, len(orders_df) + 1))

con.register("customers_df", customers_df)
con.register("orders_df", orders_df)
con.execute("CREATE OR REPLACE TABLE customers AS SELECT * FROM customers_df")
con.execute("CREATE OR REPLACE TABLE orders AS SELECT * FROM orders_df")


print(con.sql("SELECT region, COUNT(*) AS customers, ROUND(AVG(revenue), 2) AS avg_revenue FROM customers GROUP BY region ORDER BY region"))
print(con.sql("SELECT COUNT(*) AS orders, COUNT(DISTINCT customer_id) AS customers_with_orders FROM orders"))
print(f"revenue vs order count correlation: {pd.Series(n_orders).corr(customers_df['revenue']):.2f}")







┌─────────┬───────────┬─────────────┐
│ region  │ customers │ avg_revenue │
│ varchar │   int64   │   double    │
├─────────┼───────────┼─────────────┤
│ Central │       500 │      823.07 │
│ East    │       500 │      817.89 │
│ West    │      2000 │      810.39 │
└─────────┴───────────┴─────────────┘

┌────────┬───────────────────────┐
│ orders │ customers_with_orders │
│ int64  │         int64         │
├────────┼───────────────────────┤
│  20490 │                  3000 │
└────────┴───────────────────────┘

revenue vs order count correlation: 0.94


### Step 1.2: Write both queries (7 pts)

In [21]:
JOINED_ROWS = "SELECT c.customer_id, c.revenue FROM customers c JOIN orders o ON o.customer_id = c.customer_id WHERE c.region = 'West'"
PLAIN_ROWS = "SELECT customer_id, revenue FROM customers WHERE region = 'West'"

per_customer, n_plain = con.execute(f"SELECT AVG(revenue), COUNT(*) FROM ({PLAIN_ROWS})").fetchone()
per_order, n_joined = con.execute(f"SELECT AVG(revenue), COUNT(*) FROM ({JOINED_ROWS})").fetchone()
print(f"West, per customer: ${per_customer:,.2f} over {n_plain:,} rows")
print(f"West, after the join: ${per_order:,.2f} over {n_joined:,} rows")
print(f"ratio: {per_order / per_customer:.2f}")

West, per customer: $810.39 over 2,000 rows
West, after the join: $1,237.69 over 13,473 rows
ratio: 1.53


> **🛑 Verification checkpoint.** On West-region customers you should get about
> **$812** from the un-joined query (accept $780 to $845) and about **$1,240**
> from the joined one (accept $1,160 to $1,320), a ratio between **1.45 and
> 1.60**. If the two numbers are close together, your revenue is not correlated
> with order count, there is no fan-out to find, and the rest of the problem set
> has nothing to work on — go back to Step 1.1. **Do not proceed until it
> matches.**

*Your answer here.*

810.39 is the correct number. The metric we use is average revenue per customer, the unit of observation is the customer,omer and they should onl count once. This is done by the plain query as it keeps 2,000 rows for 2000 customers. The join changes it to an order, so it has 13,474 rows which does not align with the number of customers.

### Step 1.3: The rule (10 pts)

In [22]:
def rows_check(con, row_sql, key):
    """Sends warning if rows repeat, none if they do not"""
    n_rows, n_keys = con.execute(f"SELECT COUNT(*), COUNT(DISTINCT {key}) FROM ({row_sql})").fetchone()
    if n_rows > n_keys:
        return f"FAN-OUT: {n_rows:,} rows for {n_keys:,} {key} values"
    return None

print("joined:", rows_check(con, JOINED_ROWS, "customer_id"))
print("plain: ", rows_check(con, PLAIN_ROWS, "customer_id"))

joined: FAN-OUT: 13,473 rows for 2,000 customer_id values
plain:  None


*Your answer here.*
This check compares the rows with the distinct customers. If rows > customers, it means a join has repeated customers which makes the averages wrong. This check fires on the joined query, and does not on the plain one as customers and rows align.

## Phase 2: The Rows That Disappear (30 pts)

### Step 2.1: Introduce the unknown (5 pts)

In [23]:
# 15% of customers (450) were onboarded before region was mandatory
unknown_ids = rng.choice(customers_df["customer_id"].to_numpy(), size=int(0.15 * N_CUSTOMERS), replace=False)
con.register("unknown_ids", pd.DataFrame({"customer_id": unknown_ids}))
con.execute("UPDATE customers SET region = NULL WHERE customer_id IN (SELECT customer_id FROM unknown_ids)")
print(con.sql("SELECT region, COUNT(*) AS customers FROM customers GROUP BY region ORDER BY region"))

┌─────────┬───────────┐
│ region  │ customers │
│ varchar │   int64   │
├─────────┼───────────┤
│ Central │       423 │
│ East    │       421 │
│ West    │      1706 │
│ NULL    │       450 │
└─────────┴───────────┘



### Step 2.2: Lose them, then find them (15 pts)

In [24]:
n_ne = con.execute("SELECT COUNT(*) FROM customers WHERE region != 'West'").fetchone()[0]
n_meant = con.execute("SELECT COUNT(*) FROM customers WHERE region IS DISTINCT FROM 'West'").fetchone()[0]
print(f"!= 'West': {n_ne:,} rows | IS DISTINCT FROM 'West': {n_meant:,} rows | dropped by !=: {n_meant - n_ne:,}")

!= 'West': 844 rows | IS DISTINCT FROM 'West': 1,294 rows | dropped by !=: 450


*Your answer here.*
The first query returns 844 rows and the second returns 1294, because region!= West is unknown and not true for null regions, and WHERE only keeps rows where the condition is true, dropping customers with no region.

THe regional comparisons with a filter like this leave out the about 15% of regionless customers, if the customers are newer, smaller, or more likely to churn, the regional figure would only describe older, larger, and more loyal customers which biases the estimate by making average revenue too high and turnover too low in the company. The regions also will not add up to the company total.

### Step 2.3: Aggregate without collapsing (10 pts)

In [25]:
WINDOW_SQL = """
    SELECT customer_id, region, revenue,
           AVG(revenue) OVER (PARTITION BY region) AS region_avg,
           revenue - AVG(revenue) OVER (PARTITION BY region) AS vs_region
    FROM customers
    ORDER BY customer_id
"""
with_avg = con.execute(WINDOW_SQL).df()
print(with_avg.head(6).round(2).to_string(index=False))
print(f"rows in {N_CUSTOMERS:,}, rows out {len(with_avg):,}")
print(f"{(with_avg['vs_region'] > 0).sum():,} customers are above their region's average")

 customer_id  region  revenue  region_avg  vs_region
           1    West  1242.57      810.25     432.32
           2    East   659.53      828.32    -168.79
           3 Central   352.94      816.45    -463.51
           4 Central   808.80      816.45      -7.65
           5    West   886.48      810.25      76.23
           6    East  1190.58      828.32     362.26
rows in 3,000, rows out 3,000
1,109 customers are above their region's average


## Phase 3: The Causal Question (25 pts)

### Step 3.1: The programme evaluation (12 pts)

In [26]:
# program adds about $30  of revenue
TRUE_EFFECT = 30.0

n_accounts = 5_000
account_size = rng.normal(500, 150, n_accounts)
#larger accounts are more likely west
west = (rng.random(n_accounts) < 1 / (1 + np.exp(-(account_size - 500) / 75))).astype(int)
revenue = 200 + 1.2 * account_size + TRUE_EFFECT * west + rng.normal(0, 100, n_accounts)
prog = pd.DataFrame({"account_size": account_size, "west": west, "revenue": revenue})

naive = prog.loc[prog.west == 1, "revenue"].mean() - prog.loc[prog.west == 0, "revenue"].mean()
adjusted = smf.ols("revenue ~ west + account_size", data=prog).fit().params["west"]
print(f"truth ${TRUE_EFFECT:.2f} | naive ${naive:.2f} | holding account size ${adjusted:.2f}")

truth $30.00 | naive $255.89 | holding account size $30.98


*Your answer here.*

The true effect of the program is $30. The naive comparison is $255.89, which is about 8 times too high. This is because larger accounts were more likely to be in the West, and they bring in more revenue. The naive gap mixed the programs effect with that size difference. If you hold account size fixed, you get 30.98, which is close to the truth.

### Step 3.2: The control that ruins it (13 pts)

In [27]:
prog["tickets"] = rng.poisson(np.maximum(0, 10 + 6 * prog["west"] + 0.03 * (prog["revenue"] - 800)))
with_tickets = smf.ols("revenue ~ west + account_size + tickets", data=prog).fit().params["west"]
print(f"truth ${TRUE_EFFECT:.2f} | naive ${naive:.2f} | holding account size ${adjusted:.2f} | adding tickets ${with_tickets:.2f}")

truth $30.00 | naive $255.89 | holding account size $30.98 | adding tickets $-58.05


*Your answer here.*

Adding support tickets ended up making the estimate worse. It went from \$30.98 which is close to the true effect of $30, to \$-58.05. The tickets seem like a fair measure of engagement but they are driven by both the program and revenue. Account managers open tickets for the program customers, and bigger spenders file more. Among customers with the same number of tickets, program customers spend less and others spend more. Holding ticktes fixed causes programme customers to be compared to bigger spenders, which makes the program look harmful. Controls should only be for things that are not caused by the program and revenue.

## Phase 4: AI Expansion — The Co-Pilot Era (20 pts)

**Status: AI usage AUTHORISED and REQUIRED.** Use the **P.R.I.M.E. framework**.

### Task 4.1: The query reviewer (12 pts)

In [16]:
"""Grain/NULL reviewer for simple SELECT statements on DuckDB tables.

No third-party SQL parser; supports a top-level SELECT over a base table with
ordinary JOINs, optional WHERE, GROUP BY, HAVING and ORDER BY. Queries with
CTEs, derived FROM tables, comma joins or set operations receive an explicit
REVIEW_INCOMPLETE warning rather than silently passing.
"""

import re


_END_CLAUSES = re.compile(
    r"\b(?:GROUP\s+BY|ORDER\s+BY|WHERE|FROM|HAVING|WINDOW|QUALIFY|"
    r"LIMIT|OFFSET|FETCH|UNION|INTERSECT|EXCEPT)\b",
    re.IGNORECASE,
)
_AGGS = re.compile(
    r"\b(SUM|AVG|COUNT|STRING_AGG|LIST|ARRAY_AGG|MEDIAN|STDDEV(?:_SAMP|_POP)?|"
    r"VARIANCE|VAR_SAMP|VAR_POP)\s*\(\s*(?!DISTINCT\b)",
    re.IGNORECASE,
)
_IDENT = r"[A-Za-z_][A-Za-z_0-9]*"
_BASE = re.compile(r"^\s*FROM\s+(" + _IDENT + r"(?:\." + _IDENT + r")?)\b", re.I)
_RESERVED = {
    "join", "inner", "left", "right", "full", "outer", "cross", "natural",
    "semi", "anti", "asof", "on", "using", "where", "group", "order", "limit",
}


def _mask_literals(s):
    """Preserve offsets but blank comments/quoted strings/identifiers."""
    out = list(s)
    i = 0
    while i < len(s):
        if s.startswith("--", i) or s.startswith("/*", i):
            end = s.find("\n", i + 2) if s.startswith("--", i) else s.find("*/", i + 2)
            end = len(s) if end < 0 else (end if s.startswith("--", i) else end + 2)
            for j in range(i, end):
                out[j] = " "
            i = end
            continue
        if s[i] in ("'", '"'):
            quote = s[i]
            j = i + 1
            while j < len(s):
                if s[j] == quote:
                    if j + 1 < len(s) and s[j + 1] == quote:
                        j += 2
                        continue
                    j += 1
                    break
                j += 1
            for k in range(i, j):
                out[k] = " "
            i = j
            continue
        i += 1
    return "".join(out)


def _clauses(sql):
    """Find *top-level* SQL clauses; nested WHERE/FROM do not split the query."""
    mask = _mask_literals(sql)
    depth = 0
    found = []
    for token in re.finditer(r"\(|\)|" + _END_CLAUSES.pattern, mask, re.I):
        if token.group() == "(":
            depth += 1
        elif token.group() == ")":
            depth -= 1
        elif depth == 0:
            found.append((re.sub(r"\s+", " ", token.group().upper()), token.start(), token.end()))
    return mask, found


def _warning(code, message, **details):
    return {"code": code, "severity": "warning", "message": message,
            "details": details}



def _nullable_comparison_columns(con, source_mask, base_table, base_alias, predicate):
    """Best-effort *schema-level* risk check for straightforward comparisons.

    Runtime UNKNOWN counting below is the definitive check on current data.
    This scan also catches nullable predicates before the first NULL arrives.
    """
    relations = [(base_table, base_alias)]
    join_pattern = re.compile(r"\bJOIN\s+(" + _IDENT + r"(?:\." + _IDENT +
                              r")?)\b(?:\s+(?:AS\s+)?(" + _IDENT + r")\b)?", re.I)
    for m in join_pattern.finditer(source_mask):
        t, a = m.group(1), m.group(2)
        relations.append((t, a if a and a.lower() not in _RESERVED else t.split(".")[-1]))

    cols = []
    for table, alias in relations:
       rows = con.execute("PRAGMA table_info('" + table.replace("'", "''") + "')").fetchall()
       for row in rows:
            cols.append((alias, row[1], not bool(row[3] or row[5])))
    pred = _mask_literals(predicate)
    findings = []
    for alias, name, nullable in cols:
        if not nullable:
            continue
        qualified = rf"\b{re.escape(alias)}\.{re.escape(name)}\b"
        bare = rf"(?<!\.)\b{re.escape(name)}\b"
        unique_name = sum(cname.lower() == name.lower() for _, cname, _ in cols) == 1
        candidates = [qualified] + ([bare] if unique_name else [])
        for ref in candidates:
            comparison = (ref + r"\s*(?:=|<>|!=|<=|>=|<|>|\b(?:IN|LIKE|ILIKE|BETWEEN|NOT\s+IN)\b)")
            if re.search(comparison, pred, re.I) and not re.search(
                ref + r"\s+IS\s+NOT\s+NULL\b", pred, re.I
            ):
                findings.append(f"{alias}.{name}")
                break
    return sorted(set(findings))


def review_query(con, sql):
    """Return structured correctness warnings; never run the supplied SELECT.

    DuckDB connection `con` must already contain customers and orders.
    This is a grain-risk check, not a general-purpose SQL proof or parser.
    """
    sql = sql.strip()
    masked, clauses = _clauses(sql)
    if not re.match(r"^\s*SELECT\b", masked, re.I) or ";" in masked.strip().rstrip(";"):
        return [_warning("REVIEW_INCOMPLETE", "Expected one SELECT statement (no CTE or multiple statements).")]
    sql = sql.rstrip(";").rstrip()
    masked, clauses = _clauses(sql)

    # EXPLAIN validates the original query without fetching dashboard results.
    try:
        con.execute("EXPLAIN " + sql.rstrip(";")).fetchall()
    except Exception as e:
        return [_warning("INVALID_QUERY", "DuckDB could not plan this SELECT.", error=str(e))]

    from_parts = [x for x in clauses if x[0] == "FROM"]
    if len(from_parts) != 1 or any(x[0] in ("UNION", "INTERSECT", "EXCEPT") for x in clauses):
        return [_warning("REVIEW_INCOMPLETE", "Set operations or FROM-less SELECTs are not supported.")]

    _, from_at, from_end = from_parts[0]
    after_from = next((start for _, start, _ in clauses if start > from_at), len(sql))
    source = sql[from_at:after_from].strip()
    source_mask = _mask_literals(source)
    base = _BASE.match(source_mask)
    if not base or re.search(r"\bJOIN\s*\(", source_mask, re.I):
        return [_warning("REVIEW_INCOMPLETE", "FROM/JOIN subqueries or quoted table names are not supported.")]

    # A comma at top-level means an implicit cross join, which is outside scope.
    depth = 0
    for ch in source_mask:
        depth += (ch == "(") - (ch == ")")
        if ch == "," and depth == 0:
            return [_warning("REVIEW_INCOMPLETE", "Comma joins are not supported.")]
    if re.search(r"\b(?:FULL|RIGHT|ASOF)\s+(?:OUTER\s+)?JOIN\b", source_mask, re.I):
        return [_warning("REVIEW_INCOMPLETE", "FULL, RIGHT and ASOF joins need a different grain probe.")]
    table = base.group(1)
    tail = source_mask[base.end():].lstrip()
    alias_match = re.match(r"(?:AS\s+)?(" + _IDENT + r")\b", tail, re.I)
    alias = (alias_match.group(1) if alias_match and
             alias_match.group(1).lower() not in _RESERVED else table.split(".")[-1])

    # Prefer the declared primary key; use DuckDB's physical rowid otherwise.
    # The fallback is valid for actual base tables, not arbitrary views.
    try:
        info = con.execute("PRAGMA table_info('" + table.replace("'", "''") + "')").fetchall()
        if not info:
            raise ValueError("No base-table metadata")
        pk = sorted((r for r in info if r[5]), key=lambda r: r[5])
        if len(pk) == 1:
            key_expr = f'{alias}."{pk[0][1].replace(chr(34), chr(34)*2)}"'
        elif not any(r[1].lower() == "rowid" for r in info):
            key_expr = alias + ".rowid"
        else:
            raise ValueError("No single-column primary key and rowid is shadowed")
    except Exception as e:
        return [_warning("REVIEW_INCOMPLETE", "Could not determine base-table row identity.", error=str(e))]

    where_parts = [x for x in clauses if x[0] == "WHERE" and x[1] > from_at]
    predicate = None
    if where_parts:
        _, _, where_end = where_parts[0]
        after_where = next((start for _, start, _ in clauses if start >= where_end), len(sql))
        predicate = sql[where_end:after_where].strip()

# One pass of the raw joined relation supplies all cardinality statistics.
    # COUNT(DISTINCT base-key) reveals fan-out even when lost inner-join rows
    # happen to cancel duplicated rows in the total COUNT(*).
    # The base-table scalar COUNT runs separately, but the full JOIN runs ONCE.
    fields = [
        "COUNT(*) AS joined_rows",
        f"COUNT(DISTINCT {key_expr}) AS seen_base_rows",
        f"(SELECT COUNT(*) FROM {table}) AS base_rows",
    ]
    if predicate:
        fields += [
            f"COUNT(*) FILTER (WHERE ({predicate})) AS kept_rows",
            f"COUNT(DISTINCT {key_expr}) FILTER (WHERE ({predicate})) AS kept_base_rows",
            # WHERE retains TRUE only. FALSE and UNKNOWN (NULL) are both
            # rejected, but UNKNOWN means incomplete data, not a negative.
            # A nullable status <> 'cancelled' therefore silently excludes
            # unknown status records; this is a correctness/grain decision.
            f"COUNT(*) FILTER (WHERE ({predicate}) IS NULL) AS unknown_rows",
        ]
    probe_sql = "SELECT " + ", ".join(fields) + " " + source
    try:
        result = con.execute(probe_sql).fetchone()
    except Exception as e:
        return [_warning("REVIEW_INCOMPLETE", "Could not execute a single-pass grain probe.", error=str(e))]

    joined, seen, base_rows = (int(x) for x in result[:3])
    kept, kept_seen, unknown = ((int(x) for x in result[3:6])
                                if predicate else (joined, seen, 0))
    warnings = []
    has_join = bool(re.search(r"\bJOIN\b", source_mask, re.I))
    extra_rows = joined - seen  # per-base duplicates in INNER/LEFT join result
    missed_rows = base_rows - seen  # base rows lost by INNER join
    if has_join and (joined != base_rows or extra_rows > 0 or missed_rows > 0):
        warnings.append(_warning(
            "JOIN_CARDINALITY_CHANGE", "JOIN changes the base-table grain or row count.",
            base_table=table, base_rows=base_rows, joined_rows=joined,
            distinct_base_rows=seen, excess_join_rows=extra_rows,
            missing_base_rows=missed_rows,
        ))

    if unknown:
        warnings.append(_warning(
            "NULL_FILTER_DROPS_UNKNOWNS",
            "WHERE rejects rows for which its predicate is UNKNOWN (NULL).",
            predicate=predicate, unknown_join_rows=unknown,
        ))

    if predicate and unknown == 0:
        try:
            nullable_columns = _nullable_comparison_columns(con, source_mask, table, alias, predicate)
        except Exception:
            nullable_columns = []  # Runtime UNKNOWN check above still applies.
        if nullable_columns:
            warnings.append(_warning(
                "NULLABLE_FILTER_RISK",
                "WHERE compares nullable columns; future NULLs may be dropped as UNKNOWN.",
                predicate=predicate, nullable_columns=nullable_columns,
                unknown_join_rows_currently=0,
            ))

    select_clause = sql[masked.upper().find("SELECT") + 6:from_at]
    aggs = sorted(set(m.group(1).upper() for m in _AGGS.finditer(_mask_literals(select_clause))))
    if aggs and has_join and kept > kept_seen:
        warnings.append(_warning(
            "AGGREGATE_ON_FANOUT",
            "A duplicate-sensitive aggregate sees repeated base-table rows; verify the measure's grain.",
            aggregates=aggs, rows_after_filter=kept,
            distinct_base_rows_after_filter=kept_seen,
            extra_rows_after_filter=kept - kept_seen,
        ))
    return warnings


if __name__ == "__main__":
    import duckdb

    con = duckdb.connect(":memory:")
    con.execute("CREATE TABLE customers (customer_id INT PRIMARY KEY, budget INT)")
    con.execute("CREATE TABLE orders (order_id INT PRIMARY KEY, customer_id INT, status VARCHAR)")
    con.execute("INSERT INTO customers VALUES (1, 100), (2, 1040)")
    con.execute("INSERT INTO orders VALUES (10, 1, 'paid'), (11, 1, 'paid'), (12, 2, NULL)")

    # Illustrative queries, NOT the user's unpublished Phase 1 queries.
    demo_one = """SELECT SUM(c.budget) AS board_value
                  FROM customers c LEFT JOIN orders o
                    ON c.customer_id = o.customer_id"""
    demo_two = """SELECT SUM(c.budget) AS paid_budget
                  FROM customers c LEFT JOIN orders o
                    ON c.customer_id = o.customer_id
                  WHERE o.status <> 'cancelled'"""

    w1, w2 = review_query(con, demo_one), review_query(con, demo_two)
    assert {w["code"] for w in w1} == {"JOIN_CARDINALITY_CHANGE", "AGGREGATE_ON_FANOUT"}
    assert {w["code"] for w in w2} == {
        "JOIN_CARDINALITY_CHANGE", "NULL_FILTER_DROPS_UNKNOWNS", "AGGREGATE_ON_FANOUT"
    }
    assert con.execute(demo_one).fetchone()[0] == 1240  # Wrong if each budget is per customer.
    assert con.execute("SELECT SUM(budget) FROM customers").fetchone()[0] == 1140


In [28]:
# Run review_query on your Phase 1 and Phase 2 queries.
P1_BAD = "SELECT AVG(c.revenue) FROM customers c JOIN orders o ON o.customer_id = c.customer_id WHERE c.region = 'West'"
P1_GOOD = "SELECT AVG(revenue) FROM customers WHERE region = 'West'"
P1_SWAPPED = "SELECT AVG(c.revenue) FROM orders o JOIN customers c ON o.customer_id = c.customer_id WHERE c.region = 'West'"
P2_NE = "SELECT COUNT(*) FROM customers WHERE region != 'West'"
P2_MEANT = "SELECT COUNT(*) FROM customers WHERE region IS DISTINCT FROM 'West'"

tests = [("Phase 1 joined, as in my notebook", f"SELECT AVG(revenue), COUNT(*) FROM ({JOINED_ROWS})"),
         ("Phase 1 joined, one query", P1_BAD), ("Phase 1 plain", P1_GOOD),
         ("Phase 1 joined, tables swapped", P1_SWAPPED),
         ("Phase 2 !=", P2_NE), ("Phase 2 IS DISTINCT FROM", P2_MEANT), ("Step 2.3 window", WINDOW_SQL)]
for name, sql in tests:
    print(name, [w["code"] for w in review_query(con, sql)])

Phase 1 joined, as in my notebook ['REVIEW_INCOMPLETE']
Phase 1 joined, one query ['JOIN_CARDINALITY_CHANGE', 'NULL_FILTER_DROPS_UNKNOWNS', 'AGGREGATE_ON_FANOUT']
Phase 1 plain ['NULL_FILTER_DROPS_UNKNOWNS']
Phase 1 joined, tables swapped ['NULL_FILTER_DROPS_UNKNOWNS']
Phase 2 != ['NULL_FILTER_DROPS_UNKNOWNS']
Phase 2 IS DISTINCT FROM ['REVIEW_INCOMPLETE']
Step 2.3 window []


*Your answer here.*

It caught that the Phase 1 joined query got warnings, including aggregate on fanout on average, so it would have caught the $1240. It also caught region!=West removing the 450 NULL rows in Phase 2.

It missed my notebooks Phase 1 query that has the join inside a subquery, and it returns REVIEW_INCOMPLETE with no fanout warning. When the tables(FROM orders JOIN customers) are swapped, the inflated average passes without a warning. It also rejects the corrected Phase 2 query.

### Task 4.2: The one-page writeup (8 pts)

*Your answer here.*

Dear Director
$810 is the correct number. The metric we use is average revenue per customer, so each customer should count once, and this query does that.

The two differed because the $1240 query joined customers to their orders, so each customer was counted each time they ordered. Customers who order more also Spend more or less and pull the average to their habits. In this case, they spent more carried the most weight and pushedup the average.

We are not sure if the programme works yet. The bigger accounts were more likely to be in the West, and bigger accounts do well with or without the program. In the simulation in which the true effect was \$30, the West gap was \$256, but comparing accounts of the same size brought it back to $31. Controlling for tickets made it worse at \$-58 because they are related to program and revenue.

I would recommend that take a few steps before putting a number on a slide.
We should check that the row count equals to the number of customers before taking the average.
We should also keep customers with no region and make them their own group.

### AI Transcript

Paste **your prompt exactly as you sent it** and the model's replies **exactly as they came back**, then one line on each change you made. Name the model and the date.

*Model and date:* GPT-6 11/1


*Prompt:*

[Prep] Act as an analytics engineer who reviews SQL before it reaches
a dashboard.

[Request] Write a Python function review_query(con, sql) that runs
against DuckDB and returns a list of warnings for a given SELECT. It
should flag at minimum: a join that changes the row count, a filter on
a nullable column that will silently drop unknowns, and an aggregate
computed over a fanned-out result.

[Iterate] duckdb and pandas only. Reuse my table names, customers and
orders. Return structured warnings, not printed text, so I can test it.

[Mechanism Check] Add comments explaining how you detect fan-out
without running the full join twice, and why three-valued logic makes
the null filter a correctness issue rather than a style issue.

[Evaluate] Run it against both of my Phase 1 queries and tell me
whether it would have caught the $1,240 before it reached the board.
```

*Reply:*

I've created review_query(con, sql) as a reusable SQL quality check for DuckDB. It returns structured warning dictionaries, not printed messages, and uses only the Python standard library and DuckDB.
Download review_query.py

What it detects
Warning code	Detection
JOIN_CARDINALITY_CHANGE	Join duplicates or loses rows relative to the starting table
NULL_FILTER_DROPS_UNKNOWNS	WHERE evaluates to UNKNOWN, silently excluding rows
NULLABLE_FILTER_RISK	A nullable comparison could exclude future null values
AGGREGATE_ON_FANOUT	SUM, COUNT, AVG, etc. operate on duplicated base-table rows
REVIEW_INCOMPLETE	Query structure exceeds the reviewer's supported scope

How to use it
from review_query import review_querysql = """SELECT SUM(c.budget) AS total_budgetFROM customers cLEFT JOIN orders o    ON c.customer_id = o.customer_idWHERE o.status <> 'cancelled'"""warnings = review_query(con, sql)codes = {w["code"] for w in warnings}assert "JOIN_CARDINALITY_CHANGE" in codesassert "NULL_FILTER_DROPS_UNKNOWNS" in codesassert "AGGREGATE_ON_FANOUT" in codes



Every warning contains code, severity, message, and a details dictionary with measurements or predicates suitable for assertions.
Mechanism checks
Fan-out: The function counts joined rows and distinct starting-table identities in the same aggregate probe. It also counts the starting table separately. This reveals duplicated and missing rows without executing the complete join twice. DuckDB supports the conditional aggregate FILTER clauses used in the probe.

DuckDB
+1




Three-valued logic: SQL WHERE retains only TRUE. A comparison such as status <> 'cancelled' evaluates to UNKNOWN for null statuses, which means these rows are discarded even though they are not known to be cancelled. The reviewer measures how many rows are excluded this way.
Evaluation of the $1,240
The file includes two illustrative queries using customers and orders, with executable assertions.
Illustrative joined SUM
$1,240
One customer amount counted twice
Correct customer-grain SUM
$1,140
Each customer's amount counted once

The first illustrative query is designed to trigger JOIN_CARDINALITY_CHANGE and AGGREGATE_ON_FANOUT. The second additionally triggers NULL_FILTER_DROPS_UNKNOWNS.

Would it have caught your actual $1,240? If that figure arose from summing customer-level amounts after a one-to-many orders join, the reviewer is designed to flag the problem before dashboard release.

I cannot verify your two actual Phase 1 queries because neither their SQL nor their database is available here. DuckDB also wasn't installed in this execution environment, so I checked the probe logic with a compatible SQL engine rather than claiming a DuckDB execution. The downloadable file includes DuckDB assertions you can run directly with python review_query.py.

The reviewer is deliberately scoped to straightforward SELECT/JOIN statements. For unsupported constructions such as CTEs and derived-table joins, it returns REVIEW_INCOMPLETE instead of silently certifying a query as safe.



  

review_query.pyCode
-

## Before you submit

- [ ] Every heading above is still here, in order, with work under it.
- [ ] Every number you report is produced by a cell above it, and every claim has a text cell.
- [ ] The verification checkpoint(s) are met before anything built on them.
- [ ] Phase 4: the AI transcript is the exchange, not a summary; your prompt is in it as sent.
- [ ] Submit the Colab share link on Canvas (upload the `.ipynb` as a fallback).
- [ ] GitHub portfolio: *File › Save a copy in GitHub*, path `Problem Set 3/Econ_3916_PS3.ipynb`, with a short `README.md` in that folder.
- [ ] Run *Runtime › Restart and run all* once, top to bottom, before you hand in.